#### 1. Depedencies and libraries

In [1]:
!pip install -q bertopic sentence-transformers transformers torch pandas numpy matplotlib seaborn scikit-learn scipy tqdm Sastrawi

preventing kernel crash

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
import torch
torch.set_num_threads(1)

In [23]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from collections import Counter

In [ ]:
warnings.filterwarnings("ignore") # hides warning messages
sns.set_theme(style="whitegrid", context="notebook") # sets the theme for seaborn plots
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)


Declaring variables for config

In [5]:
csv_file = "psn_news_scraped.csv"
random_state = 42

embedding_model = "paraphrase-multilingual-MiniLM-L12-v2" # Indonesian + English
sentiment_model_id = "w11wo/indonesian-roberta-base-sentiment-classifier" # Indonesian text
sentiment_model_en = "cardiffnlp/twitter-roberta-base-sentiment" # English text

min_chars = 300 # dropping articles that's shorter than this number

#### 2. Data Loading

here loading data will be performed and to check the scrapped article or news data 

In [6]:
raw_data = pd.read_csv(csv_file)
print("Shape:", raw_data.shape)
raw_data.info()
raw_data.head(3)

Shape: (918, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   link          918 non-null    object
 1   title         918 non-null    object
 2   snippet       918 non-null    object
 3   date          918 non-null    object
 4   source        918 non-null    object
 5   article_url   824 non-null    object
 6   article_text  764 non-null    object
dtypes: object(7)
memory usage: 50.3+ KB


,link,title,snippet,date,source,article_url,article_text
0,https://news.google.com/rss/articles/CBMia0FVX...,Berita Kini - Menkomdigi Meutya Hafid: Satelit...,Berita Kini - Menkomdigi Meutya Hafid: Satelit...,"Tue, 12 May 2026 07:00:00 GMT",Kementerian Komunikasi dan Digital,https://portal.komdigi.go.id/kanal-publik/beri...,Menteri Komunikasi dan Digital Meutya Hafid me...
1,https://news.google.com/rss/articles/CBMikwFBV...,KSP Dukung Pemerataan Internet di Daerah 3T Me...,KSP Dukung Pemerataan Internet di Daerah 3T Me...,"Tue, 12 May 2026 07:00:00 GMT",Kantor Staf Presiden,https://www.ksp.go.id/ksp-dukung-pemerataan-in...,"Kepala Staf Kepresidenan, Jenderal TNI (Purn.)..."
2,https://news.google.com/rss/articles/CBMiswFBV...,"Perebutan 'Kue' Internet Satelit RI Memanas, P...","Perebutan 'Kue' Internet Satelit RI Memanas, P...","Wed, 13 May 2026 07:00:00 GMT",detikInet,https://inet.detik.com/telecommunication/d-848...,Agus Tri Haryanto - detikInet\nDirektur Utama ...


In [8]:
raw_data.isnull().sum()

link              0
title             0
snippet           0
date              0
source            0
article_url      94
article_text    154
dtype: int64

#### 3. Data Cleaning

from data loading, it shows that there are articles left that is not talking about psn and how many nulls are there on url and text. so this section mainly adresses that along side other cleaning methods.

##### Convert Date to datetime and dropping rows without text

Parse dates and drop rows without text. for example converts date text into datetime so later, plot trends can be made. also removing rows where artricle text is missing

In [10]:
df = raw_data.copy()

df["date"] = pd.to_datetime(df["date"], format="%a, %d %b %Y %H:%M:%S GMT", errors="coerce")
print("Rows with invalid date format:", df["date"].isnull().sum())

n0 = len(df)
df = df.dropna(subset=["article_text"]).copy()
print(f"Dropped {n0 - len(df)} rows with missing article_text")
print(f"{len(df)} rows left after dropping {n0 - len(df)} rows")

Rows with invalid date format: 0
Dropped 154 rows with missing article_text
764 rows left after dropping 154 rows


##### Noise removal

news or article data scraped a bunch of text from article / news content. the code cell below will explore if the article contains noise, eg ads, share buttons, copyright footers, “Baca Juga”, links.

In [24]:
counter = Counter()
for text in df["article_text"]:
    lines = set(line.strip() for line in str(text).split("\n") if line.strip())
    counter.update(lines)

noise = pd.DataFrame(counter.most_common(30), columns=["line", "n_articles"])
noise = noise[noise["n_articles"] >= 3]
noise["line"] = noise["line"].str[:100]
noise

,line,n_articles
0,SCROLL TO CONTINUE WITH CONTENT,169
1,Editor:,85
2,Penulis:,75
3,Bagikan,75
4,Salin Pranala,75
5,Agus Tri Haryanto - detikInet,66
6,Baca Juga,60
7,"Dilarang keras mengambil konten, melakukan cra...",38
8,© 2020,37
9,ADVERTISEMENT,35


based on the result, it shows that there are a lot of noise on most articles. hence it need to be removed 

In [26]:
noise_patterns = [
    r"^SCROLL TO CONTINUE", r"^advertisement$", r"^baca\s*juga", r"^bagikan$", r"^salin pranala$",
    r"^penulis\s*:", r"^editor\s*:", r"^translator\s*:", r"^pewarta", r"^can't find what",
    r"^available on", r"^ikuti saluran", r"^mau notif", r"^dapatkan aplikasi", r"tanpa suara beda",
    r"^©", r"copyright", r"all rights reserved", r"dilarang keras", r"crawling atau pengindeksan",
    r"izin tertulis dari direksi", r"^alamat\s*:", r"whatsapp\s*:", r"^address\s*:",
    r"-\s*detik\w+$",          # bylines / photo credits such as "Agus Tri Haryanto - detikInet"
    r"^foto\s*:", r"^sumber\s*:", r"^source\s*:", r"^read also", r"^also read",
]

noise_re = re.compile("|".join(noise_patterns), re.IGNORECASE)
url_re = re.compile(r"https?://\S+|www\.\S+")

In [27]:
def clean_article(text: str) -> str:
    lines = []
    for line in str(text).replace("\xa0", " ").split("\n"):
        line = line.strip()
        if not line or noise_re.search(line):
            continue
        line = url_re.sub("", line)
        line = line.replace("[…]", "").replace("[...]", "")
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            lines.append(line)
    return " ".join(lines)

In [28]:
df["text"] = df["article_text"].apply(clean_article)

df["title_clean"] = df.apply(
    lambda r: re.sub(r"\s+-\s+" + re.escape(str(r["source"])) + r"\s*$", "", str(r["title"])).strip(), axis=1
)

ex = df.sample(1, random_state=1).iloc[0]

print("BEFORE:", ex["article_text"][:400].replace("\n", " | "))
print()
print("AFTER :", ex["text"][:400])

BEFORE: Agus Tri Haryanto - detikInet | Dalam hitungan tiga hari lagi, Indonesia akan memiliki satelit internet pertama. Satelit Republik Indonesia atau Satria-1 dijadwalkan meluncur dari Cape Canaveral, Florida, Amerika Serikat, pada 18 Juni 2023 waktu setempat atau 19 Juni 2023 waktu Indonesia. | Sebelumnya satelit ini dirakit oleh Thales Alenia Space di Prancis sejak September 2020 hingga Mei 2023. Rencana

AFTER : Dalam hitungan tiga hari lagi, Indonesia akan memiliki satelit internet pertama. Satelit Republik Indonesia atau Satria-1 dijadwalkan meluncur dari Cape Canaveral, Florida, Amerika Serikat, pada 18 Juni 2023 waktu setempat atau 19 Juni 2023 waktu Indonesia. Sebelumnya satelit ini dirakit oleh Thales Alenia Space di Prancis sejak September 2020 hingga Mei 2023. Rencana awal, akan diangkut dari Pran


##### Dropping short articles

some pages returned only short texts, eg copyright footer, " read more ", or a cookie warnings. so those short texts will be dropped as well

In [29]:
n0 = len(df)
df["n_chars"] = df["text"].str.len()
df = df[df["n_chars"] >= min_chars].copy()
print(f"Dropped {n0 - len(df)} rows with text shorter than {min_chars} characters")

Dropped 90 rows with text shorter than 300 characters


##### Removing duplicates

google news often returns the same article from different links, so those dudplicates will be removed.

In [30]:
n0 = len(df)
df = df.drop_duplicates(subset="text").drop_duplicates(subset="title_clean").copy()
print(f"Dropped {n0 - len(df)} duplicate rows based on text and title_clean")

Dropped 2 duplicate rows based on text and title_clean


##### PSN titles only

from the result of scrapping, there are some articles that are not about psn that has been scraped. so it needs to be removed. the article will be kept only if psn is in the title or psn mentioned at least twice in the content.

In [31]:
psn_re = re.compile(
    r"\bPSN\b|Pasifik Satelit|Satelit Nusantara|Nusantara\s+(Satu|Dua|Tiga|Lima|One|Two|Three|Five)",
    flags=re.IGNORECASE,
)

min_mentions = 2

n0 = len(df)
df["psn_mentions"] = df["text"].apply(lambda t: len(psn_re.findall(t)))
in_title = df["title_clean"].apply(lambda t: bool(psn_re.search(t)))

df = df[in_title | (df["psn_mentions"] >= min_mentions)].copy()
print(f"Dropped {n0 - len(df)} rows that do not mention PSN at least {min_mentions} times in the text or in the title")

Dropped 330 rows that do not mention PSN at least 2 times in the text or in the title


##### Distribution of languages for articles

since the data is mixed between Indonesian and English, we need to know the distribution of languages to pick the right model.

In [32]:
ID = re.compile(r"\b(yang|dan|di|untuk|dengan|dari|pada|ini|itu|dalam|akan)\b", re.IGNORECASE)
EN = re.compile(r"\b(the|and|of|to|in|for|with|that|is|are)\b", re.IGNORECASE)

def detect_lang(text: str) -> str:
    sample = text[:3000]
    return "en" if len(EN.findall(sample)) > len (ID.findall(sample)) else "id"

df["lang"] = df["text"].apply(detect_lang)
df["lang"].value_counts()

lang
id    285
en     57
Name: count, dtype: int64

##### Helper columns and reset index

After cleaning, adding helper columns (`year`, `year_month`, `n_words`) for EDA charts, sort the articles by date, and reset the index so the rows doesnt have empty gaps. `doc_id` column gives each article a unique ID, which is used to match topic and sentiment results to the right article.

In [33]:
df["year"] = df["date"].dt.year
df["year_month"] = df["date"].dt.to_period("M").dt.to_timestamp()
df["n_words"] = df["text"].str.split().str.len()

df = df.sort_values("date").reset_index(drop=True)
df["doc_id"] = df.index

print("Cleaned dataset:", df.shape)
df[["doc_id", "date", "source", "lang", "title_clean", "n_words"]].head(10)

Cleaned dataset: (342, 16)


,doc_id,date,source,lang,title_clean,n_words
0,0,2017-05-17 07:00:00,Liputan6.com,id,Satelit Nano Pertama Buatan Mahasiswa Indonesi...,283
1,1,2017-08-22 07:00:00,SpaceNews,en,Back-to-back commercial satellite wins leave C...,1389
2,2,2017-09-07 07:00:00,Kompas.com,id,Anomali Satelit Telkom 1,430
3,3,2017-12-05 08:00:00,kumparan,id,PSN Bikin Satelit Baru untuk Sebar Internet di...,538
4,4,2018-07-09 07:00:00,Kompas.id,id,"Agustus 2018, Telkom Luncurkan Satelit Telkom 4",609
5,5,2018-09-11 07:00:00,GeekWire,en,Israeli moon lander to ride SpaceX rocket in S...,691
6,6,2019-01-23 08:00:00,Bisnis Tekno,id,Satelit Kapasitas Jumbo Nusantara Satu Mengorb...,258
7,7,2019-01-23 08:00:00,Suara.com,id,PSN Siap Luncurkan Satelit dengan Kapasitas Br...,308
8,8,2019-01-23 08:00:00,kumparan,id,Satelit Nusantara Satu PSN Meluncur Februari 2...,399
9,9,2019-01-25 08:00:00,ANTARA News Jateng,id,Satelit Nusantara Satu buatan Indonesia segera...,355
